In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd

StatementMeta(, , -1, Cancelled, , Cancelled, True)

# Bronze Data Extraction

In [ ]:
df = spark.table('bronze.students')

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
df.summary().show()

StatementMeta(, , -1, Cancelled, , Cancelled, True)

# Data Cleaning

In [ ]:
df = df.dropDuplicates()

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
df.summary().show()

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
cols_to_clean = df.columns
clean_expr = r'[^a-zA-Z0-9\s@.\-]'
for c in cols_to_clean:
    if df.schema[c].dataType.simpleString() == 'string':
        df = df.withColumn(c, regexp_replace(col(c), clean_expr, ""))

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
df.describe().toPandas()

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
df.filter(
    (col("Student_ID").isNull()) |
    (col("First_Name").isNull()) |
    (col("Last_Name").isNull()) |
    (col("Email").isNull()) |
    (col("Gender").isNull()) |
    (col("Department").isNull()) |
    (col("Grade").isNull()) |
    (col("Extracurricular_Activities").isNull()) |
    (col("Internet_Access_at_Home").isNull()) |
    (col("Parent_Education_Level").isNull()) |
    (col("Family_Income_Level").isNull()) |
    (col("Age").isNull()) |
    (col("Math_Score").isNull()) |
    (col("assi_Math").isNull()) |
    (col("Science_Score").isNull()) |
    (col("assi_Science").isNull()) |
    (col("English_Score").isNull()) |
    (col("assi_English").isNull()) |
    (col("Social_Studies_Score").isNull()) |
    (col("assi_Social_Studie").isNull()) |
    (col("French_Score").isNull()) |
    (col("assi_French").isNull()) |
    (col("Computer_Science_Score").isNull()) |
    (col("assi_Computer_Science").isNull()) |
    (col("last_score").isNull()) |
    (col("Study_Hours_per_Week").isNull()) |
    (col("Stress_Level_1_10").isNull()) |
    (col("Sleep_Hours_per_Night").isNull()) |
    (col("assi_late").isNull()) |
    (col("non_academic_attendance").isNull()) |
    (col("academic_attendance").isNull()) |
    (col("lms_login_freq_per_day").isNull()) |
    (col("lms_active_avg_hrs").isNull()) |
    (col("resource_access_avg_hrs").isNull())
).toPandas()

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
null_counts = df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])

display(null_counts)

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
df = df.withColumn("assi_late", ceil(col("assi_late")))

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
df = df.withColumn(
    "Grade",
    when(col("last_score").isNull(), col("Grade"))
    .when(col("last_score") >= 88, "A")
    .when(col("last_score") >= 75, "B")
    .when(col("last_score") >= 65, "C")
    .when(col("last_score") >= 50, "D")
    .otherwise("F")
)

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
display(df)

StatementMeta(, , -1, Cancelled, , Cancelled, True)

# Standartion

In [ ]:
result = df.groupBy("Gender", "Grade", "Department") \
  .agg(
      avg("assi_late").alias("avg_assi_late"),
      avg("non_academic_attendance").alias("avg_non_academic"),
      avg("academic_attendance").alias("avg_academic"),
      avg("lms_login_freq_per_day").alias("avg_login"),
      avg("lms_active_avg_hrs").alias("avg_lms_hours"),
      avg("resource_access_avg_hrs").alias("avg_resource")
  ) \
  .orderBy("Grade", "Department")

display(result)

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
max_result = df.groupBy("Gender", "Grade", "Department") \
  .agg(
      max("assi_late").alias("max_assi_late"),
      max("non_academic_attendance").alias("max_non_academic"),
      max("academic_attendance").alias("max_academic"),
      max("lms_login_freq_per_day").alias("max_login"),
      max("lms_active_avg_hrs").alias("max_lms_hours"),
      max("resource_access_avg_hrs").alias("max_resource")
  ) \
  .orderBy("Grade", "Department")

display(max_result)

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
min_result = df.groupBy("Gender", "Grade", "Department") \
  .agg(
      min("assi_late").alias("min_assi_late"),
      min("non_academic_attendance").alias("min_non_academic"),
      min("academic_attendance").alias("min_academic"),
      min("lms_login_freq_per_day").alias("min_login"),
      min("lms_active_avg_hrs").alias("min_lms_hours"),
      min("resource_access_avg_hrs").alias("min_resource")
  ) \
  .orderBy("Grade", "Department")

display(min_result)

StatementMeta(, , -1, Cancelled, , Cancelled, True)

# Data Imputation

In [ ]:
joined_df = df.join(result, ["Gender", "Grade", "Department"], "left")

columns_to_fix = {
    "assi_late": "avg_assi_late",
    "non_academic_attendance": "avg_non_academic",
    "academic_attendance": "avg_academic",
    "lms_login_freq_per_day": "avg_login",
    "lms_active_avg_hrs": "avg_lms_hours",
    "resource_access_avg_hrs": "avg_resource"
}

final_df = joined_df
for original_col, avg_col in columns_to_fix.items():
    final_df = final_df.withColumn(
        original_col, 
        coalesce(col(original_col), col(avg_col))
    )

final_df = final_df.drop(*result.columns) 

display(final_df)

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [ ]:
final_df = final_df.withColumn("assi_late", ceil(col("assi_late")))

In [ ]:
df_with_id = final_df.withColumn("id", monotonically_increasing_id())

In [ ]:
df_with_id = df_with_id.select("id", *[c for c in df_with_id.columns if c != 'id'])

display(df_with_id)

In [ ]:
df_with_id = df_with_id.withColumn("Math_Score", round(col("Math_Score"), 2)) \
    .withColumn("Science_Score", round(col("Science_Score"), 2)) \
    .withColumn("English_Score", round(col("English_Score"), 2)) \
    .withColumn("Social_Studies_Score", round(col("Social_Studies_Score"), 2)) \
    .withColumn("French_Score", round(col("French_Score"), 2)) \
    .withColumn("Computer_Science_Score", round(col("Computer_Science_Score"), 2)) \
    .withColumn("assi_Math", round(col("assi_Math"), 2)) \
    .withColumn("assi_Science", round(col("assi_Science"), 2)) \
    .withColumn("assi_English", round(col("assi_English"), 2)) \
    .withColumn("assi_Social_Studie", round(col("assi_Social_Studie"), 2)) \
    .withColumn("assi_French", round(col("assi_French"), 2)) \
    .withColumn("assi_Computer_Science", round(col("assi_Computer_Science"), 2)) \
    .withColumn("last_score", round(col("last_score"), 2)) \
    .withColumn("Study_Hours_per_Week", round(col("Study_Hours_per_Week"), 2)) \
    .withColumn("Stress_Level_1_10", round(col("Stress_Level_1_10"), 2)) \
    .withColumn("Sleep_Hours_per_Night", round(col("Sleep_Hours_per_Night"), 2)) \
    .withColumn("assi_late", ceil(col("assi_late"))) \
    .withColumn("non_academic_attendance", ceil(col("non_academic_attendance"))) \
    .withColumn("academic_attendance", ceil(col("academic_attendance"))) \
    .withColumn("lms_login_freq_per_day", round(col("lms_login_freq_per_day"), 2)) \
    .withColumn("lms_active_avg_hrs", round(col("lms_active_avg_hrs"), 2)) \
    .withColumn("resource_access_avg_hrs", round(col("resource_access_avg_hrs"), 2))

# Formating

In [ ]:
for column in df_with_id.columns:
    if df_with_id.schema[column].dataType.simpleString() == 'string':
        df_with_id = df_with_id.withColumn(column, lower(col(column)))

# Save to Delta

In [ ]:
df_with_id.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.silver_Student_")